# 2장 05. Great Expectations API 기초 실습

이 notebook은 Great Expectations(GE)를 처음 보는 사람을 위한 가장 작은 예제입니다. 실제 과정 데이터에 들어가기 전에, 작은 표 하나로 GE API가 어떤 순서로 움직이는지 확인합니다.


## 1. 작은 예제 데이터 만들기

### 1-1. 검사할 표 준비

먼저 환자 4명의 작은 표를 만듭니다. 일부러 문제가 있는 값을 넣어 둡니다.

- `label`에 허용되지 않은 값 `unknown`이 있습니다.
- `heart_rate`에 비어 있는 값이 있습니다.
- `oxygen_saturation`에 100을 넘는 값이 있습니다.

GE는 이런 문제를 사람이 눈으로 찾기 전에 규칙으로 확인하게 해 주는 도구입니다.


In [ ]:
import contextlib
import io

import great_expectations as gx
import pandas as pd
from IPython.display import clear_output
from great_expectations.expectations import (
    ExpectColumnToExist,
    ExpectColumnValuesToBeBetween,
    ExpectColumnValuesToBeInSet,
    ExpectColumnValuesToNotBeNull,
)

sample_dataframe = pd.DataFrame(
    {
        "patient_id": [1, 2, 3, 4],
        "label": ["high_risk", "low_risk", "unknown", "high_risk"],
        "heart_rate": [80, None, 125, 72],
        "oxygen_saturation": [98, 95, 103, 92],
    }
)

display(sample_dataframe)


**출력 확인**

표는 4행짜리 작은 데이터입니다. GE를 쓰기 전에는 사람이 직접 보고 문제를 찾아야 합니다.

여기서는 눈으로도 `unknown`, 비어 있는 `heart_rate`, `oxygen_saturation=103`을 찾을 수 있습니다. 하지만 실제 데이터가 수천 행, 수만 행이 되면 사람이 매번 확인하기 어렵습니다. GE는 이런 확인을 검사 항목으로 만들어 반복 실행하게 해 줍니다.


## 2. GE 작업 공간 만들기

### 2-1. Context 만들기

`Context`는 GE 작업의 시작점입니다. 쉽게 말하면 “이번 GE 검사를 관리하는 작업 공간”입니다.

이 셀에서는 `gx.get_context(mode="ephemeral")`을 사용합니다. 여기서 `mode`는 GE 작업 공간을 어디에 어떻게 보관할지 정하는 옵션입니다.

| mode | 쉬운 뜻 | 언제 쓰나 | 이 강의에서의 위치 |
| --- | --- | --- | --- |
| `ephemeral` | 메모리에만 있는 임시 작업 공간 | 작은 실습, 테스트, 노트북에서 빠르게 확인할 때 | 이번 실습에서 사용 |
| `file` | 로컬 파일로 저장되는 작업 공간 | 검사 기준과 결과를 프로젝트 폴더에 계속 보관할 때 | 알아만 둠 |
| `cloud` | GE Cloud에 연결하는 작업 공간 | 조직 단위 관리형 환경에서 쓸 수 있던 방식 | 강의 범위 밖 |

공식 문서 기준으로 `ephemeral`은 현재 Python 세션이 끝나면 유지되지 않는 임시 Context이고, `file`은 설정과 메타데이터를 파일로 저장하는 Context입니다. `cloud` 모드는 API에는 남아 있지만 현재 GX Cloud 종료로 이 강의에서는 다루지 않습니다.

따라서 여기서는 파일을 만들지 않는 임시 작업 공간을 씁니다. GE API 흐름을 연습하기에 가장 부담이 적습니다.


In [ ]:
context = gx.get_context(mode="ephemeral")

context_summary = pd.DataFrame(
    [
        {
            "api": "gx.get_context(mode='ephemeral')",
            "created_object": type(context).__name__,
            "easy_meaning": "GE 임시 작업 공간을 만들었습니다.",
        }
    ]
)

display(context_summary)


**출력 확인**

`EphemeralDataContext`가 보이면 임시 GE 작업 공간이 만들어진 것입니다.

`ephemeral`은 “임시”라는 뜻입니다. 이 기초 실습에서는 설정 파일이나 문서를 저장하지 않고, 메모리 안에서만 GE API 흐름을 확인합니다.


## 3. 데이터를 GE에 연결하기

### 3-1. Datasource, Data Asset, Batch 만들기

GE는 바로 DataFrame을 검사하지 않고, 데이터를 몇 단계로 감싸서 다룹니다.

| 단계 | 코드 | 쉬운 뜻 |
| --- | --- | --- |
| Datasource | `add_pandas()` | Pandas 데이터를 쓰겠다고 알려 줍니다. |
| Data Asset | `add_dataframe_asset()` | 검사할 데이터 묶음의 이름을 붙입니다. |
| Batch Definition | `add_batch_definition_whole_dataframe()` | DataFrame 전체를 한 번에 검사하겠다고 정합니다. |
| Batch | `get_batch()` | 이번 실행에서 실제로 검사할 데이터를 넘깁니다. |


In [ ]:
datasource = context.data_sources.add_pandas(name="pandas_basic")
data_asset = datasource.add_dataframe_asset(name="sample_vital_signs")
batch_definition = data_asset.add_batch_definition_whole_dataframe(name="whole_dataframe")
batch = batch_definition.get_batch(batch_parameters={"dataframe": sample_dataframe})

api_object_summary = pd.DataFrame(
    [
        {"step": "1", "api": "context.data_sources.add_pandas", "object_name": datasource.name, "easy_meaning": "Pandas 데이터를 쓰는 연결"},
        {"step": "2", "api": "datasource.add_dataframe_asset", "object_name": data_asset.name, "easy_meaning": "검사할 데이터 묶음"},
        {"step": "3", "api": "data_asset.add_batch_definition_whole_dataframe", "object_name": batch_definition.name, "easy_meaning": "DataFrame 전체를 검사하는 방식"},
        {"step": "4", "api": "batch_definition.get_batch", "object_name": type(batch).__name__, "easy_meaning": "이번에 실제로 검사할 데이터"},
    ]
)

display(api_object_summary)


**출력 확인**

이 출력은 GE가 데이터를 어떤 이름으로 관리하는지 보여 줍니다.

중요한 것은 마지막 줄의 `Batch`입니다. GE에서 검사는 결국 Batch를 대상으로 실행됩니다. 즉, `sample_dataframe`이 GE 안에서는 `Batch`라는 검사 대상이 된 것입니다.


## 4. 검사 항목 만들기

### 4-1. Expectation Suite 만들기

`Expectation`은 검사 항목입니다. `Expectation Suite`는 검사 항목을 모아 둔 검사표입니다.

이번 예제에서는 네 가지 검사 항목을 만듭니다.

| 검사 항목 | 쉬운 질문 |
| --- | --- |
| `ExpectColumnToExist` | 이 컬럼이 있나요? |
| `ExpectColumnValuesToBeInSet` | 값이 허용 목록 안에 있나요? |
| `ExpectColumnValuesToNotBeNull` | 값이 비어 있지 않나요? |
| `ExpectColumnValuesToBeBetween` | 값이 정해진 범위 안에 있나요? |


In [ ]:
expectation_suite = gx.ExpectationSuite(name="basic_data_check")

expectation_plan = pd.DataFrame(
    [
        {"expectation": "ExpectColumnToExist", "column": "label", "rule": "label 컬럼이 있어야 합니다."},
        {"expectation": "ExpectColumnValuesToBeInSet", "column": "label", "rule": "label 값은 high_risk 또는 low_risk여야 합니다."},
        {"expectation": "ExpectColumnValuesToNotBeNull", "column": "heart_rate", "rule": "heart_rate는 비어 있으면 안 됩니다."},
        {"expectation": "ExpectColumnValuesToBeBetween", "column": "oxygen_saturation", "rule": "oxygen_saturation은 0부터 100 사이여야 합니다."},
    ]
)

display(expectation_plan)


**출력 확인**

아직 검사를 실행한 것은 아닙니다. 지금은 “어떤 규칙으로 검사할지” 계획을 세운 단계입니다.

GE에서는 이 규칙 하나하나를 `Expectation`이라고 부릅니다. 여러 Expectation을 하나로 묶으면 `Expectation Suite`가 됩니다.


In [ ]:
expectations = [
    ExpectColumnToExist(column="label"),
    ExpectColumnValuesToBeInSet(
        column="label",
        value_set=["high_risk", "low_risk"],
        result_format="SUMMARY",
    ),
    ExpectColumnValuesToNotBeNull(
        column="heart_rate",
        result_format="SUMMARY",
    ),
    ExpectColumnValuesToBeBetween(
        column="oxygen_saturation",
        min_value=0,
        max_value=100,
        result_format="SUMMARY",
    ),
]

for expectation in expectations:
    expectation_suite.add_expectation(expectation)

suite_summary = pd.DataFrame(
    [
        {
            "order": index + 1,
            "expectation_type": expectation.expectation_type,
            "column": expectation.dict().get("column"),
            "easy_meaning": expectation_plan.loc[index, "rule"],
        }
        for index, expectation in enumerate(expectation_suite.expectations)
    ]
)

display(suite_summary)


**출력 확인**

`expectation_type`은 GE API의 실제 검사 이름입니다. 이름이 길어 보여도 구조는 단순합니다.

- `expect_column...`: 컬럼을 검사합니다.
- `to_exist`: 존재해야 합니다.
- `to_be_in_set`: 허용 목록 안에 있어야 합니다.
- `to_not_be_null`: 비어 있으면 안 됩니다.
- `to_be_between`: 범위 안에 있어야 합니다.

이제 검사표가 준비되었습니다.


## 5. 검사 실행하기

### 5-1. Validator 만들기

`Validator`는 Batch와 Expectation Suite를 연결해서 실제 검사를 실행하는 객체입니다.

쉽게 말하면 다음 두 가지를 합치는 단계입니다.

- 무엇을 검사할 것인가: `batch`
- 어떤 기준으로 검사할 것인가: `expectation_suite`


In [ ]:
validator = context.get_validator(
    batch=batch,
    expectation_suite=expectation_suite,
)

validator_summary = pd.DataFrame(
    [
        {"object": "batch", "easy_meaning": "검사 대상 데이터", "value": type(batch).__name__},
        {"object": "expectation_suite", "easy_meaning": "검사 기준 묶음", "value": expectation_suite.name},
        {"object": "validator", "easy_meaning": "검사 대상과 기준을 연결한 실행 객체", "value": type(validator).__name__},
    ]
)

display(validator_summary)


**출력 확인**

`Validator`가 만들어졌다면 검사 실행 준비가 끝난 것입니다.

이 단계까지는 아직 통과/실패를 확인하지 않았습니다. 다음 셀에서 `validate()`를 호출해야 실제 검사 결과가 나옵니다.


In [ ]:
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    validation_result = validator.validate(result_format="SUMMARY")

clear_output(wait=True)

validation_overview = pd.DataFrame(
    [
        {"item": "overall_success", "value": validation_result.success, "easy_meaning": "전체 검사 통과 여부"},
        {"item": "evaluated_expectations", "value": validation_result.statistics["evaluated_expectations"], "easy_meaning": "실행한 검사 항목 수"},
        {"item": "successful_expectations", "value": validation_result.statistics["successful_expectations"], "easy_meaning": "통과한 검사 항목 수"},
        {"item": "unsuccessful_expectations", "value": validation_result.statistics["unsuccessful_expectations"], "easy_meaning": "실패한 검사 항목 수"},
    ]
)

display(validation_overview)


**출력 확인**

`overall_success`가 `False`이면 전체 검사 중 하나 이상이 실패했다는 뜻입니다.

이번 예제는 일부러 문제 있는 값을 넣었기 때문에 실패가 정상입니다. 실패가 나왔다는 사실보다 더 중요한 것은 “어떤 검사에서 왜 실패했는지”를 다음 표에서 확인하는 것입니다.


In [ ]:
def compact_unexpected_values(values: list[object]) -> str:
    if not values:
        return ""
    visible_values = []
    for value in values[:5]:
        if pd.isna(value):
            visible_values.append("<missing>")
        else:
            visible_values.append(str(value))
    return ", ".join(visible_values)

result_rows = []
for result in validation_result.results:
    kwargs = result.expectation_config.kwargs
    result_detail = result.result
    result_rows.append(
        {
            "expectation_type": result.expectation_config.type,
            "column": kwargs.get("column", ""),
            "success": result.success,
            "unexpected_count": result_detail.get("unexpected_count", 0),
            "unexpected_percent": round(result_detail.get("unexpected_percent", 0), 2),
            "sample_unexpected_values": compact_unexpected_values(result_detail.get("partial_unexpected_list", [])),
        }
    )

validation_result_table = pd.DataFrame(result_rows)

display(validation_result_table)


**출력 확인**

이 표가 GE 검사의 핵심 결과입니다.

- `success=True`: 해당 검사 항목은 통과했습니다.
- `success=False`: 해당 검사 항목은 실패했습니다.
- `unexpected_count`: 규칙을 어긴 값의 개수입니다.
- `sample_unexpected_values`: 규칙을 어긴 값의 예시입니다.

이번 예제에서는 다음처럼 해석합니다.

| 실패한 컬럼 | 실패 이유 | 출력에서 보는 값 |
| --- | --- | --- |
| `label` | 허용 목록에 없는 값이 있습니다. | `unknown` |
| `heart_rate` | 비어 있는 값이 있습니다. | `<missing>` |
| `oxygen_saturation` | 0부터 100 사이가 아닌 값이 있습니다. | `103` |

이제 GE API 흐름을 한 문장으로 요약할 수 있습니다.

```text
Context를 만들고, 데이터를 Batch로 연결한 뒤, Expectation Suite를 만들어 validate()로 검사한다.
```


## 6. 2-2 실습과 연결하기

### 6-1. 이 기초 예제가 2-2에서 어떻게 이어지는가

2장 06 `06_great_expectations_lab.ipynb`는 이 기초 예제와 같은 흐름을 더 실제적인 데이터에 적용합니다.

| 기초 노트북 | 2-2 노트북 |
| --- | --- |
| 작은 `sample_dataframe`을 직접 만듭니다. | `vital_signs_valid_degraded.csv`를 읽습니다. |
| 검사 항목 4개를 코드에서 직접 만듭니다. | `great_expectations_rules.yaml`에서 검사 기준을 읽습니다. |
| 결과를 화면 표로 확인합니다. | 결과를 JSON, Markdown, HTML 저장 파일로 남깁니다. |
| GE API 흐름을 이해합니다. | 데이터 품질 실패가 모델 평가 해석에 주는 제한을 정리합니다. |

따라서 GE를 처음 본다면 이 notebook을 먼저 실행하고, 그다음 `06_great_expectations_lab.ipynb`로 이동하면 됩니다.
